# TerraDelta offline prediction

Reads AIF_INPUT_DIR and writes AIF_PREDICTION_PATH. No training, network calls or submit commands.


In [ ]:
# Offline inference. All assets are bundled in this ZIP.
import os
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "assets/config.yaml").is_file():
    raise FileNotFoundError("Run predict.ipynb from the extracted submission root")
sys.path.insert(0, str(ROOT / "assets/code"))
import torch
from terradelta.utils.io import load_config
from terradelta.inference.predictor import predict_directory

CONFIG = load_config(ROOT / "assets/config.yaml")
INPUT_DIR = Path(os.environ.get("AIF_INPUT_DIR", "./input"))
PREDICTION_PATH = Path(os.environ.get("AIF_PREDICTION_PATH", "./prediction.csv"))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# CPU fallback is limited to accelerator readiness, before processing input.
from terradelta.models.factory import build_model
from terradelta.models.checkpoint import load_checkpoint
if DEVICE == "cuda":
    try:
        probe = build_model({"encoder_weights": None}).to(DEVICE).eval()
        load_checkpoint(ROOT / "assets/model/model.pt", probe)
        with torch.inference_mode():
            probe(torch.zeros(1, 6, 256, 256, device=DEVICE))
        del probe
    except (RuntimeError, torch.cuda.OutOfMemoryError) as error:
        print("CUDA readiness failed; using CPU:", type(error).__name__)
        DEVICE = "cpu"
    torch.cuda.empty_cache()
rows = predict_directory(INPUT_DIR, PREDICTION_PATH,
    ROOT / "assets/model/model.pt", CONFIG, device=DEVICE)
print("Saved", len(rows), "pairs to", PREDICTION_PATH)
